# 01 古典的な機械学習 — 特徴量を足すと、基準を超えられるか

> 🔰 **やさしい導入**（この回のひとこと）
>
> - **ひとことで言うと**: 過去の発電量・天気などを特徴量として、木の手法（ランダムフォレスト・勾配ブースティング）で予測する回です。
> - **たとえ話**: 「日射量は多いか？」「夏か？」と質問を重ねて、答えにたどり着く。
> - **先に読むと分かりやすい**: [00_basics.ipynb](00_basics.ipynb) の「2-5（機械学習とは）」。用語が分からなくなったら、同じNotebookの「用語集」へ。
> - **難しければ**: 各コードの後ろにある「**出力の見方**」の文章だけ読めば、結論は追えます。

> 🔒 すべて**合成データ**です。⏱ 所要時間: 説明 15分 ＋ 実行 約2分

## ゴール
- **ラグ特徴量**（過去の発電量・天気・気温など）を作り、**リークしない**ことを確かめる。
- **リッジ回帰・ランダムフォレスト・勾配ブースティング**で、00 の基準（予報の直線回帰）を超えられるかを見る。
- **どの特徴量が効いているか**を調べ、「予報の誤差」が支配的であることを理解する。
- **予報が完璧**な診断用の世界で、過去の並びから**隠れた状態**（温度・よごれ）を読めるかを確かめる。

## 前のNotebookとのつながり
00 では、予報の直線回帰（L=1でMAE 0.875）が、学習を使わない基準の中で最良でした。上限は、予報が完璧でも 0.636、隠れた状態まで知れば 0.194。**この間のどこまで、特徴量と機械学習で詰められるか**が、ここでの問いです。

## 特徴量の作り方（リークしない）
予測の起点は「対象日の **L日前の夜**」。使ってよいのは **起点日までの観測** と、**対象日の予報 fcL**、**対象日の日付**だけです。
| グループ | 中身 |
|---|---|
| base | 予報 fcL、季節（sin, cos） |
| gen | 起点日までの発電量（ラグ0〜6、7日・30日平均）、**性能比**（発電量 ÷ (0.85×日射の観測)。温度とよごれの影響が出る） |
| weather | 起点日の天気・日照率・日射・気温、気温の7日平均 |
| dry | 雨の降らない日の連続日数、直近14日の雨の割合（よごれの手がかり） |

## キーワード
- **ラグ特徴量**: 過去の値を、特徴量にしたもの。
- **リーク**: 予測時点で分からない情報（未来・答え）が入力に混ざること。見かけの精度が上がるが、使い物にならない。
- **勾配ブースティング（HGB）**: 木を少しずつ足していく手法。表データで強い。
- **並べ替え重要度**: ある特徴量の列をシャッフルすると、誤差がどれだけ増えるか。大きいほど、その特徴量に頼っている。

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import run_phase1
from tsf_models import build_features
from tsf_synth import make_series

plt.rcParams["font.family"] = ["Hiragino Sans", "AppleGothic", "DejaVu Sans"]
pd.set_option("display.width", 200)

# リークの確認: 起点日より後の観測を書き換えても、特徴量が変わらないか
df = make_series(1_000_000, 4)
L, t = 3, 1000
X = build_features(df, L)
d2 = df.copy()
after = d2.index > t - L
keep = d2.loc[t, f"fc{L}"]
for c in ["gen_kwh", "gsr", "temp", "sunshine_ratio", "gsr_true", "soiling"]:
    d2.loc[after, c] = 999.0
d2.loc[after, f"fc{L}"] = 999.0
d2.loc[t, f"fc{L}"] = keep
print("特徴量は同じか:", X.loc[t].equals(build_features(d2, L).loc[t]))
X.loc[t].round(2)

**出力の見方**: 起点日（t−3）より後の観測を、でたらめな値（999）に書き換えても、対象日 t の特徴量は**変わりません**（`True`）。**対象日の観測が特徴量に入っていない**証拠です（予報 fcL だけは、使ってよい情報として残しています）。この確認はテストにも入れてあります。

In [ ]:
e1 = run_phase1.e1()
e1

**出力の見方**（L=1・検証・10世界の平均MAE）:
- **勾配ブースティング**は、**予報と季節だけ（base）で 0.813**。基準（直線回帰 0.875）を、すでに **7%** 上回ります。理由: 予報と**季節**の**非線形な組み合わせ**（夏は温度の影響で、同じ予報でも発電量が低い）を、木が捉えるため。
- **リッジ**は、base で 0.865（直線回帰とほぼ同じ）。**線形のまま特徴量を足しても**、ほとんど変わりません（0.865→0.853）。
- **特徴量を足していく効果は小さい**: HGBは 0.813 → 0.807（＋gen）→ 0.810（＋weather）→ 0.804（＋dry）。標準誤差は 0.008〜0.009 なので、**足したグループの効果は、ほとんど区別できません**。
- 過去の発電量・性能比・雨の降らない日数を足しても、**期待したほど良くならない**。なぜか。次以降で調べます。

In [ ]:
e2 = run_phase1.e2()
piv = e2.pivot(index="手法", columns="リードL", values="MAE")
order = ["予報の直線回帰", "リッジ", "ランダムフォレスト", "勾配ブースティング", "上限1（予報が完璧）", "上限2（隠れた状態まで）"]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for name in order:
    g = e2[e2["手法"] == name]
    ax[0].plot(g["リードL"], g["MAE"], "o-", label=name)
ax[0].set_xlabel("リード L（日）"); ax[0].set_ylabel("検証のMAE"); ax[0].legend(fontsize=7); ax[0].grid(alpha=0.3); ax[0].set_title("手法の比較（10世界の平均）")
for name in ["リッジ", "ランダムフォレスト", "勾配ブースティング"]:
    g = e2[e2["手法"] == name]
    ax[1].errorbar(g["リードL"], g["直線回帰との差"], yerr=2 * g["差の標準誤差"], fmt="o-", capsize=3, label=name)
ax[1].axhline(0, color="k", lw=0.5); ax[1].set_xlabel("リード L（日）"); ax[1].set_ylabel("直線回帰との差（負=良い。誤差棒±2標準誤差）"); ax[1].legend(); ax[1].grid(alpha=0.3)
ax[1].set_title("基準に対する改善")
plt.tight_layout(); plt.show()
piv.loc[order]

**出力の見方**（全特徴量・10世界の平均MAE）:
- **ランダムフォレスト**が最良: L=1で **0.785**（直線回帰 0.875 に対して −0.090、標準誤差0.004。約10%改善）。HGB 0.804（−0.071）、リッジ 0.853（−0.022）。
- 改善の大きさは、リードで変わります: ランダムフォレストの改善は L=1で −0.090、L=3で −0.212、L=5で −0.077（標準誤差 0.013）。相対的には、L=1の10%、L=3の12%、L=5の**2.8%**。**リードが長く、予報が不正確になると、予報の誤差が支配的**で、手法を変えても詰められない。
- **上限1（予報が完璧）0.636** と **上限2 0.194** には、**まだ遠い**。L=1でRFの 0.785 は、上限1との差が 0.15 もあります。
- 特徴量を増やし、手法を高度にしても、**予報の誤差が壁**になっている。

In [ ]:
e3, grid = run_phase1.e3()
e3

**出力の見方**（HGBのハイパーパラメータを、1つの世界のvalで選び、別の9世界で評価）:
- 36通り（深さ4×学習率3×木の数3）のうち、選んだ世界で最良は **深さ6・学習率0.02・木300本**（MAE 0.802）。
- 別の9世界では、**選んだ設定 0.789**、既定（深さ4・0.05・300）**0.801**。差は 0.012 で、標準誤差（約0.009）と同程度。**チューニングの効果は、小さい**。
- 最悪の設定（深さ2・学習率0.02・木100本）は **1.052** と大きく悪化する。**木が足りない・浅すぎる**と、非線形を捉えられない。
- 選んだ世界のvalのMAE（0.802）は、別の世界の値（0.789）とほぼ同じ。**この程度のばらつきは、世界による違い**です。

In [ ]:
e4 = run_phase1.e4()
e4

**出力の見方**（HGB・L=1・特徴量のグループをシャッフルしたときのMAEの増加）:
- **base（予報・季節）**: **+4.37**。ほとんどの情報は、ここにあります。シャッフルすると、予測が成り立たなくなる。
- ほかのグループは小さい: gen（過去の発電量）+0.022、性能比 +0.008、weather +0.038、dry +0.025。**それぞれ、MAEを0.01〜0.04動かす程度**。
- 予報の誤差が大きい（L=1でも約0.9 kWh換算）ので、**過去の観測が持つ微かな信号（よごれ・温度）は、予報のノイズに埋もれて**います。

In [ ]:
e5 = run_phase1.e5()
e5

**出力の見方**（**診断用**。予報を真の日射量に置き換えた、現実にはありえない世界。L=1）:
- 予報が完璧だと、base だけでも HGB は **0.284**。ここから特徴量を足すと: ＋gen **0.233** → ＋weather 0.229 → ＋dry **0.218**。**上限2（0.194）に近づく**。
- つまり、**隠れた状態は、過去の並びから読める**。ただし**予報が良いとき**に限る。実際の設定（予報の誤差あり）では、その信号が埋もれていた（E1）。
- リッジ（線形）は 0.300 → 0.263 と、あまり改善しない。**非線形（温度の低下・よごれの掛け算）を捉えるのは、木のほう**が得意。
- 示唆: **隠れた状態を読む価値は、他の誤差が小さいときに出る**。Phase3（系列モデル）では、この「予報が良い世界」と「予報が悪い世界」の両方で、系列モデルが木を超えるかを調べます。

## 試してみよう（演習）
1. **特徴量を1グループずつ除く**: `fit_predict("hgb", tr, va, 1, ("base", "gen"))` のように、グループを変えて、MAEを比べてください。分かるのは、どのグループが無くても困らないかです。
2. **ランダムフォレストの木の深さ**: `fit_predict("rf", ..., min_samples_leaf=20)` にして、MAEがどう変わるか見てください。分かるのは、葉を大きくする（単純にする）効果です。
3. **リークをわざと作る**: `build_features` の中の `gen.shift(L + k)` を `gen.shift(k)`（L=3のとき、起点日より後の値が入る）に変えて、MAEが不自然に小さくなることを確かめてください（元に戻すこと）。分かるのは、リークの危険さです。
4. **隠れた状態を消す**: `make_series(..., soiling_rate=0.0)` の世界で E5 と同じ実験をして、dry・性能比の効果がどう変わるか見てください。

## まとめ
- **勾配ブースティングは、予報と季節だけで基準を7%上回る**（非線形）。ランダムフォレストは全特徴量で **10%**（0.875→0.785）。
- **過去の発電量・天気・気温・雨の降らない日数**を足しても、ほとんど良くならない（予報の誤差に埋もれる）。
- 予報が不正確なリードでは、**手法を変えても詰められない**。
- **予報が完璧**なら、過去の並びから隠れた状態を読める（0.284→0.218、上限0.194）。
- **ハイパーパラメータの調整**は小さな効果。**リークしない特徴量**が前提。

次の Phase2 では、**ニューラルネット（MLP）** を同じ特徴量で学習させ、木と比べます。PyTorchの導入を確認します。